In [ ]:
"""
Explain the module.
"""

In [ ]:
# Standard library
import os
from datetime import datetime

# Third-party
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader, Docx2txtLoader


load_dotenv()

### LOAD DOCUMENTS AND CREATE PANDAS DATAFRAME

In [ ]:
# Load documents with DirectoryLoader
txt_loader = DirectoryLoader("./documents", glob="**/*.txt", loader_cls=TextLoader)
pdf_loader = DirectoryLoader("./documents", glob="**/*.pdf", loader_cls=PyPDFLoader)
docx_loader = DirectoryLoader("./documents", glob="**/*.docx", loader_cls=Docx2txtLoader)

documents = txt_loader.load() + pdf_loader.load() + docx_loader.load()
print(f"Loaded {len(documents)} documents")

In [ ]:
# Print one of the documents
print(documents[0])

In [ ]:
# Create pandas dataframe from documents and metadata
rows = []
for doc in documents:
    rows.append({
        "text": doc.page_content,
        **doc.metadata # unpack metadata
    })

df = pd.DataFrame(rows)
df.head()

### OPTIONS FOR ADDING METADATA

In [ ]:
## OPTIONAL STEP
# Drop columns
df = df.drop(["list_columns_to_dorp"], axis=1)
df.head()

In [ ]:
## OPTIONAL STEP
# Columns to keep - remove unwanted metadata columns
df = df[["text", "source"]]
df.head()

In [ ]:
## OPTIONAL STEP
# Extract filename and folder from source path
df["filename"] = df["source"].apply(lambda x: os.path.basename(x))
df["folder"] = df["source"].apply(lambda x: os.path.dirname(x))
df.head()

In [ ]:
## OPTIONAL STEP
# Add file type
df["file_type"] = df["filename"].apply(lambda x: os.path.split(".")[-1])
df.head()

In [ ]:
## OPTIONAL STEP
# Add text length
df["char_count"] = df["text"].apply(len)
df["word_count"] = df["text"].apply(lambda x: len(x.split()))
df.head()

In [ ]:
## OPTIONAL STEP
# Add custom tags - example: category
def classify_doc(text):
    if "finance" in text.lower():
        return "finance"
    return "general"

df["category"] = df["text"].apply(classify_doc)
df.head()

In [ ]:
## OPTIONAL STEP - BUG !
# Add ingestion timestamp
df["ingested_at"] = datetime.utcnow()
df.head()

##!! Must make it str or int or something that is accepted during embedding

In [ ]:
## OPTIONAL STEP
# Save df for temporary storage as csv, save for reproducibility
df.to_csv("documents_metadata_df.csv", index=False)

# Load file later
df = pd.read_csv("documents_metadata_df.csv")

In [ ]:
## OPTIONAL STEP - BUG !
# Save df for temporary storage as parquet file, save for reproducibility
df.to_parquet("documents_with_metadata.parquet", index=False)

# Load file later
df = pd.read_parquet("document_with_metadata.parquet")

### CONVERT BACK FROM PANDAS DATAFRAME TO DOCUMENTS

In [ ]:
# Convert back to documents with metadata
documents_metadata = []

for _, row in df.iterrows(): 
    metadata = row.drop("text").to_dict()

    documents_metadata.append(
        Document(
            page_content=row["text"],
            metadata=metadata
        )
    )

print(f"Loaded {len(documents)} documents")

In [ ]:
# Print one of the documents
print(documents_metadata[0])

In [ ]:
## OPTIONAL STEP
# If small tweaks are needed, change metadata after document creation, example: category
for doc in documents_metadata:
    doc.metadata["category"] = "new_value"

### CREATE CHUNKS AND ADD CHUNK METADATA

In [ ]:
# Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents_metadata)

In [ ]:
# Print chunk metadata
for i, chunk in enumerate(chunks):
    print(chunk.metadata)

In [ ]:
# Print specific chunk metadata, example: source
for i, chunk in enumerate(chunks):
    print(chunk.metadata["source"])

In [ ]:
# Add chunk metadata, example: chunk_id
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i


In [ ]:
# Add chunk metadata, example: real position tracking
start = 0

for i, chunk in enumerate(chunks):
    text = chunk.page_content
    end = start + len(text)

    chunk.metadata["start_char"] = start
    chunk.metadata["end_char"] = end

    start = end

In [ ]:
# Print chunk metadata
for i, chunk in enumerate(chunks):
    print(chunk.metadata)

In [ ]:
# Print specific chunk metadata, example: source
for i, chunk in enumerate(chunks):
    print(chunk.metadata["source"])

### STANDARDIZE CHUNK METADATA

In [ ]:
# Definition to standardize chunk metadata
from pydantic_models import ChunkMetadata

def standardize_chunk_metadata(chunk):
    """
    Doc-string.
    """
    # Explain, example: "chunk_id": "3" > Pydantic converts it > 3 (int)
    std_meta = ChunkMetadata.model_validate(chunk.metadata)

    # model_dump() converts pydandic model into a standard python dict
    structured_metadata = std_meta.model_dump() 

    # Extracts "extra metadata", everything that is NOT in the schema
    extra_metadata = {key: value for key, value in chunk.metadata.items() if key not in structured_metadata}

    return chunk, structured_metadata, extra_metadata

In [ ]:
# Run standardize_chun_metadata() function on all chunks
clean_chunks = []

for chunk in chunks:
    _, structured_metadata, extra_metada = standardize_chunk_metadata(chunk)

    clean_chunk = chunk.model_copy()
    clean_chunk.metadata = {**structured_metadata, **extra_metada}

    clean_chunks.append(clean_chunk)

### INGEST CHUNKS: CHROMA DB AND SQLite FTS

In [ ]:
# Create / load Chroma DB (persisted locally)
vector_store = Chroma(
    collection_name="document_collection_1",
    embedding_function=OpenAIEmbeddings(),
    persist_directory="./chroma_db"
)

# Add chunks to Chroma db
vector_store.add_documents(clean_chunks)

print("Embeddings stored in ./chroma_db")

In [ ]:
# Initiate FTS and index chunks
from fts_search import FTSStore

fts_store = FTSStore()
fts_store.add_documents(clean_chunks)

print("Indexing complete")

### CHEK DOCUMENTS, CHUNKS, AND METADATA

In [ ]:
# View what is stored in persisted vectorstore/db
vectorstore = Chroma(
    collection_name="document_collection_1",
    embedding_function=OpenAIEmbeddings(),
    persist_directory="./chroma_db" # saved on disk
)

print(vectorstore.get())

In [ ]:
# Print chunk metadata
for i, chunk in enumerate(clean_chunks):
    print(chunk.metadata)

In [ ]:
# Print and inspect metadata
collection = vectorstore._collection
data = collection.get(include=["embeddings", "documents", "metadatas"])

print(f"Number of chunks: {len(data['documents'])}\n")

for i in range(len(data["documents"])):
    print("ID:", data["ids"][i])
    print("TEXT:", data["documents"][i])
    print("METADATA:", data["metadatas"][i])
    print("VECTOR (first 5 dims):", data["embeddings"][i][:5])
    print("-"*40)

In [ ]:
# Test retrieval, metatdata
results = vectorstore.similarity_search("your query", k=1)

for result in results:
    print(result.page_content)
    print(result.metadata)

In [ ]:
# Run PCA to see how documents cluster
data = collection.get(include=["embeddings"])
embeddings = np.array(data["embeddings"])

pca = PCA(n_components=2)
reduced = pca.fit_transform(embeddings)

In [ ]:
# Print agent graph, example: agent_compiled
from IPython.display import Image, display
display(Image(agent_compiled.get_graph().draw_mermaid_png()))